In [1]:
# Import libraries for text processing,
# dimensionality reduction, classification, and validation

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Load the preprocessed and reduced dataset
# This is a subset of the Sentiment140 corpus with texts and labels

df_reduced = pd.read_csv('twitter_reduced_Dataset.csv', encoding='latin-1')

# Split the predictor variables (texts) and the target variable (sentiment)
X = df_reduced['text'].values
y = df_reduced['target'].values

# Define a custom list of stopwords
# These words will be filtered during vectorization because they do not provide relevant information

custom_stopwords = [
    'as', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
    'is', 'it', 'for', 'with', 'that', 'this', 'was', 'be',
    'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
    'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not'
]


In [2]:
i=0
# Parameters to explore
k_values = [10]
C_values = [1]

# Initialize the results
results = []

# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Loop over the different values of parameter C
for C in C_values:
    aucs = []

    for train_idx, test_idx in cv.split(X, y):
        texts_train = X[train_idx]
        texts_test = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        # TF-IDF vectorization fitted only on the training set
        vectorizer = TfidfVectorizer(max_features=10000, stop_words=custom_stopwords)
        X_train_vect = vectorizer.fit_transform(texts_train)
        X_test_vect = vectorizer.transform(texts_test)

        # Training the logistic regression model
        model = LogisticRegression(
            penalty='l2',
            C=C,
            solver='lbfgs',
            max_iter=100,
            random_state=42,
        )
        model.fit(X_train_vect, y_train)
        print("Iteration: "+str(i) )

        # Evaluation with AUC-ROC
        probas = model.predict_proba(X_test_vect)[:, 1]
        auc = roc_auc_score(y_test, probas)
        aucs.append(auc)
        print("AUC-ROC: Fold " + str(i) +  ":" + str(auc))  # Print the AUC score for the current fold
        i=i+1
    results.append({'C': C, 'mean_auc_roc': np.mean(aucs)})

Iteration: 0
AUC-ROC: Fold 0:0.8216382812500002
Iteration: 1
AUC-ROC: Fold 1:0.8308244140624998
Iteration: 2
AUC-ROC: Fold 2:0.8233740234375
Iteration: 3
AUC-ROC: Fold 3:0.827321875
Iteration: 4
AUC-ROC: Fold 4:0.816437109375


In [3]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=10000):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=10000):

 C  mean_auc_roc
 1        0.8239
